# IT 630 Week 3 Participation: Same JSON, Three Tables

An API gives you JSON. Turning it into a table forces a choice, and the key question is not "how do I flatten this?" The key question is: **what should one row mean?**

This Colab asks you to turn one USGS GeoJSON response into three different tables:

1. one row per earthquake event
2. one row per calendar day
3. one row per magnitude category

All three are valid. They answer different questions and throw away different information.

**The habit to build:** before each transform, predict the row count and state the grain. After each transform, check the actual shape and write what the table can and cannot claim.


## 1. Get one USGS response

Run this cell as-is first. It uses a fixed Los Angeles-area query so everyone starts with the same response. Once you have completed one full run with the provided values, you may replace the latitude, longitude, radius, dates, or magnitude floor.


In [ ]:
import requests
import pandas as pd

URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format": "geojson",
    "latitude": 34.05,
    "longitude": -118.24,
    "maxradiuskm": 300,
    "starttime": "2026-05-20",
    "endtime": "2026-05-27",
    "minmagnitude": 1.0,
}

response = requests.get(URL, params=params, timeout=20)
print("status code:", response.status_code)
print("content type:", response.headers.get("Content-Type"))

response.raise_for_status()
data = response.json()
features = data.get("features", [])

print("top-level keys:", list(data.keys()))
print("feature count:", len(features))


**Answer 1:** A `200 OK` response means the request succeeded. Does it prove the response contains useful earthquake records? Use the feature count above in your answer.


## 2. Inspect one earthquake object

The top-level response is not the table. The possible rows live inside `features`, and each feature has nested parts.


In [ ]:
one = features[0]
print("feature keys:", list(one.keys()))
print("property keys sample:", list(one.get("properties", {}).keys())[:12])
print("geometry:", one.get("geometry"))


**Answer 2:** Where do these values live in the nested object?

- magnitude
- place
- time
- longitude
- latitude
- depth

Write the access path for each one, for example `feature["properties"]["mag"]`.


## 3. Table A: one row per earthquake event

This is the simplest defensible table for an event-level question: "what earthquakes happened in this query window?"

Before running the code, write your prediction.

**Prediction A:**

- one row = one `_____`
- expected rows = `_____`
- fields kept = `_____`
- fields ignored = `_____`


In [ ]:
def build_event_table(features):
    rows = []
    for feature in features:
        props = feature.get("properties") or {}
        geom = feature.get("geometry") or {}
        coords = geom.get("coordinates") or []

        lon = coords[0] if len(coords) > 0 else None
        lat = coords[1] if len(coords) > 1 else None
        depth = coords[2] if len(coords) > 2 else None

        rows.append({
            "event_id": feature.get("id"),
            "place": props.get("place"),
            "magnitude": props.get("mag"),
            "time_ms": props.get("time"),
            "longitude": lon,
            "latitude": lat,
            "depth_km": depth,
        })

    table = pd.DataFrame(rows)
    table["event_time_utc"] = pd.to_datetime(table["time_ms"], unit="ms", utc=True)
    table["event_date"] = table["event_time_utc"].dt.date
    return table

event_table = build_event_table(features)
print("event_table shape:", event_table.shape)
event_table.head()


In [ ]:
print("Missing values per event-table column:")
print(event_table.isna().sum())


**Answer 3:** Did your predicted row count match the actual row count? State the grain of `event_table` in one sentence.

**Answer 4:** GeoJSON coordinates are positional: `[longitude, latitude, depth]`. What would go wrong if you treated `coords[0]` as latitude and `coords[1]` as longitude?


## 4. Table B: one row per calendar day

Now change the question. Instead of asking what events happened, ask: "which days had the most earthquake activity?"

This table intentionally throws away event-level detail. Note: `event_date` is the **UTC** date, not the local California date.

**Prediction B:**

- one row = one `_____`
- expected rows = `_____`
- what detail will be lost? `_____`


In [ ]:
daily_table = (
    event_table
    .groupby("event_date", dropna=False)
    .agg(
        event_count=("event_id", "count"),
        max_magnitude=("magnitude", "max"),
        mean_magnitude=("magnitude", "mean"),
        mean_depth_km=("depth_km", "mean"),
    )
    .reset_index()
)

print("daily_table shape:", daily_table.shape)
daily_table.head(10)


**Answer 5:** Did your predicted row count match? If not, what did you forget to account for?

**Answer 6:** Name one claim `daily_table` supports better than `event_table`, and one claim it can no longer support because event-level detail was aggregated away.


## 5. Table C: one row per magnitude category

Change the question again: "how many events were small, moderate, or large within this query window?"

The category boundaries are modeling choices. They are not facts handed to you by the API.


In [ ]:
bins = [-float("inf"), 2.5, 4.5, float("inf")]
labels = ["small: <2.5", "moderate: 2.5 to <4.5", "large: >=4.5"]

event_table["magnitude_category"] = pd.cut(
    event_table["magnitude"],
    bins=bins,
    labels=labels,
    right=False,
)

magnitude_table = (
    event_table
    .groupby("magnitude_category", observed=False)
    .agg(
        event_count=("event_id", "count"),
        min_magnitude=("magnitude", "min"),
        max_magnitude=("magnitude", "max"),
    )
    .reset_index()
)

print("magnitude_table shape:", magnitude_table.shape)
magnitude_table


**Answer 7:** What is the grain of `magnitude_table`?

**Answer 8:** The bins `<2.5`, `2.5 to <4.5`, and `>=4.5` are choices. How could different bin cutoffs change the story this table appears to tell?

**Answer 9:** Should zero-count categories remain visible in a table like this, or should they be dropped? Explain your answer.


## 6. Compare the three tables

Fill this in after all three tables run.


In [ ]:
comparison = pd.DataFrame({
    "table": ["event_table", "daily_table", "magnitude_table"],
    "shape": [event_table.shape, daily_table.shape, magnitude_table.shape],
    "grain": [
        "one row per earthquake event",
        "one row per calendar day",
        "one row per magnitude category",
    ],
})
comparison


**Answer 10:** Which table has the most rows, and why?

**Answer 11:** Which table lost the most detail? Be specific about what it lost.

**Answer 12:** Which table is easiest to overclaim from? Name the overclaim.


## 7. Mini table card

Choose one of the three tables and complete this card. This is the main participation deliverable.

**Table chosen:** `_____`

**Grain:** one row = one `_____`

**Source query:** location, radius, date window, and magnitude floor from the params cell.

**Fields used:** `_____`

**Fields ignored or dropped:** `_____`

**Known limits:** `_____`

**One claim this table supports:** `_____`

**One claim this table does not support:** `_____`

**Most dangerous modeling choice:** `_____`


## Wrap-up

The same JSON became three different tables. None of them is the default correct table. Each table answers a different question, changes the grain, and sets a different claim boundary.

**The one line to carry:** flattening is not cleanup. It is a modeling decision.
